# SWE-Pocket — Reproduction Notebook

Reproduces every quantitative claim in *SWE-Pocket: A Repository-Size-Stratified
Evaluation Resource for Consumer-Hardware Coding Agents* without a GPU or a
model run.

Inputs (released in this repo):
- `data/stratification.csv` — structural metrics, large stratum (127 graphs)
- `data/compact_structure.csv` — structural metrics, compact stratum (22 graphs)
- `data/joined_structure_gold.csv` — structure x gold-patch x test-set, 129 rows
- `data/compact_joined.json` — same join for the compact stratum
- `resources/strata_table.json` — the shipped summary table

The competition's `tasks.jsonl` itself is not redistributed; the derived join
table above is what this notebook reads.


## 0. Setup

In [ ]:
import os, csv, json, statistics
from collections import Counter

# Resolve repo root whether the notebook is run from notebooks/ or the root
ROOT = os.getcwd() if os.path.isdir("data") else os.path.abspath("..")
P = lambda *p: os.path.join(ROOT, *p)

LARGE = list(csv.DictReader(open(P("data/joined_structure_gold.csv"))))
COMPACT = json.load(open(P("data/compact_joined.json")))
STRAT = {r["instance"]: r for r in csv.DictReader(open(P("data/stratification.csv")))}
print("large tasks:", len(LARGE), "| compact tasks:", len(COMPACT),
      "| total:", len(LARGE) + len(COMPACT))

## 1. The resource — 129 provided + 24 mined = 153 tasks
Per-repo counts, as reported in §3.

In [ ]:
print("Large stratum (provided):", dict(Counter(r["repo"] for r in LARGE)))
print("Compact stratum (mined):  ", dict(Counter(r["repo"] for r in COMPACT)))
assert len(LARGE) == 129 and len(COMPACT) == 24

## 2. Table 1 — repository structure by stratum
Means per repository; churn is the median gold-patch churn. This is the three-axis
result (§4.2).

In [ ]:
def m(rows, k): return statistics.mean(float(r[k]) for r in rows)
def med(rows, k): return statistics.median(float(r[k]) for r in rows)

def row(repo, stratum, rows):
    return dict(repo=repo, stratum=stratum, n=len(rows),
                nodes=round(m(rows, "nodes")), edges=round(m(rows, "edges")),
                density=round(m(rows, "density"), 2),
                max_depth=round(m(rows, "max_depth")),
                churn=round(med(rows, "gold_churn_total")))

table = []
for repo in ["fastapi", "rich", "requests", "httpx"]:
    table.append(row(repo, "large", [r for r in LARGE if r["repo"] == repo]))
for repo in ["packaging", "parso", "prettytable", "python-dotenv"]:
    table.append(row(repo, "compact", [r for r in COMPACT if r["repo"] == repo]))

hdr = f"{'repo':16}{'stratum':9}{'n':>4}{'nodes':>8}{'edges':>8}{'density':>9}{'depth':>7}{'churn':>7}"
print(hdr); print("-" * len(hdr))
for t in table:
    print(f"{t['repo']:16}{t['stratum']:9}{t['n']:>4}{t['nodes']:>8}{t['edges']:>8}"
          f"{t['density']:>9}{t['max_depth']:>7}{t['churn']:>7}")

### 2.1 Size ranges (the stratification is genuine)
Large 711–4,597 nodes vs compact 75–606.

In [ ]:
ln = [int(r["nodes"]) for r in LARGE]
cn = [int(r["nodes"]) for r in COMPACT]
print(f"large nodes:   {min(ln)}–{max(ln)}")
print(f"compact nodes: {min(cn)}–{max(cn)}")
assert max(cn) < min(ln), "strata must be disjoint in size"

## 3. Table 2 — structure is (largely) independent of fix size (§4.3)
Pearson correlation of each structural axis with gold-patch churn, over all 129
large tasks.

In [ ]:
import math
def pearson(xs, ys):
    n = len(xs); mx = sum(xs)/n; my = sum(ys)/n
    cov = sum((a-mx)*(b-my) for a, b in zip(xs, ys))
    vx = sum((a-mx)**2 for a in xs); vy = sum((b-my)**2 for b in ys)
    return cov / math.sqrt(vx*vy) if vx*vy else 0.0

for axis in ["nodes", "edges", "density", "max_depth"]:
    xs = [float(r[axis]) for r in LARGE]
    ys = [float(r["gold_churn_total"]) for r in LARGE]
    print(f"corr({axis:10}, churn) = {pearson(xs, ys):+.3f}")

### 3.1 The typical task is small and single-file

In [ ]:
sf = [r for r in LARGE if int(r["gold_files"]) == 1]
churn = [int(r["gold_churn_total"]) for r in LARGE]
print(f"single-source-file tasks: {len(sf)}/{len(LARGE)} = {len(sf)/len(LARGE):.0%}")
print(f"median gold churn (all):        {statistics.median(churn):.0f} lines")
print(f"median gold churn (single-file): {statistics.median(int(r['gold_churn_total']) for r in sf):.0f} lines")

### 3.2 The honest caveat — outliers and the single-file subset
Reported in §4.3: the pooled correlation is ~0, but it is weak-not-absent.

In [ ]:
xs = [float(r["nodes"]) for r in sf]; ys = [float(r["gold_churn_total"]) for r in sf]
print(f"corr(nodes, churn) | full population = {pearson([float(r['nodes']) for r in LARGE],[float(r['gold_churn_total']) for r in LARGE]):+.3f}")
print(f"corr(nodes, churn) | single-file     = {pearson(xs, ys):+.3f}")
out = sorted(LARGE, key=lambda r: -int(r["gold_churn_total"]))[:3]
print("largest fixes (outliers):")
for r in out:
    print(f"  {r['instance_id']:18} {r['gold_churn_total']:>6} lines, {r['gold_files']} files")

## 4. Test sets recovered from provided `test_patch` (§3.3)
FAIL_TO_PASS / PASS_TO_PASS totals.

In [ ]:
f2p = sum(int(r["f2p_count"]) for r in LARGE)
p2p = sum(int(r["p2p_count"]) for r in LARGE)
with_f2p = sum(1 for r in LARGE if int(r["f2p_count"]) > 0)
print(f"tasks with >=1 FAIL_TO_PASS: {with_f2p}/{len(LARGE)}")
print(f"total FAIL_TO_PASS functions: {f2p}")
print(f"total PASS_TO_PASS functions: {p2p}")

## 5. Cross-validate the shipped stratum table
Confirms `resources/strata_table.json` matches this notebook's computation.

In [ ]:
shipped = {t["repo"]: t for t in json.load(open(P("resources/strata_table.json")))}
recomputed = {t["repo"]: t for t in table}
for repo, s in shipped.items():
    r = recomputed[repo]
    assert (s["nodes"], s["edges"], s["max_depth"], s["median_churn"]) == \
           (r["nodes"], r["edges"], r["max_depth"], r["churn"]), (repo, s, r)
print("strata_table.json matches recomputation:", len(shipped), "repos")

## Summary
Every number in the paper's §4 is reproduced above from public artifacts, with no
GPU and no model run. Run order: `scripts/mine_compact.py` →
`scripts/build_compact_graphs.py` → `scripts/stratify.py` (both strata) →
`scripts/extract_tests.py` → `scripts/merge_gold_structure.py` → this notebook.